In [1]:
import pandas as pd
import io
from google.colab import files

print("Pilih file WIKIDATA (Utama) dan DBPEDIA (Pendukung):")
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Pilih file WIKIDATA (Utama) dan DBPEDIA (Pendukung):


Saving DBpedia.csv to DBpedia.csv
Saving Wikidata.csv to Wikidata.csv
User uploaded file "DBpedia.csv" with length 48816 bytes
User uploaded file "Wikidata.csv" with length 18061 bytes


In [2]:
import pandas as pd
def baca_csv_bandel(nama_file):
    # Baca mentahan
    df = pd.read_csv(nama_file, sep=',', quotechar='"', on_bad_lines='skip')

    # pecah kolom manual
    if len(df.columns) == 1:
        header_asli = df.columns[0].replace('"', '').split(',')
        df = df.iloc[:, 0].str.split(',', expand=True)
        # Ambil kolom sebanyak jumlah header yang kita dapet
        df = df.iloc[:, :len(header_asli)]
        df.columns = header_asli

    # Bersihkan nama kolom dari spasi atau tanda kutip
    df.columns = [c.replace('"', '').strip() for c in df.columns]
    return df

# 2. Eksekusi Baca File
print("--- Memproses Wikidata ---")
df_wiki = baca_csv_bandel('Wikidata.csv')
print("Kolom Wikidata yang ditemukan:", df_wiki.columns.tolist())

print("\n--- Memproses DBpedia ---")
df_dbpedia = baca_csv_bandel('DBpedia.csv')
print("Kolom DBpedia yang ditemukan:", df_dbpedia.columns.tolist())

# 3. PROSES JOIN
try:
    # Pastikan kolom kunci ada
    # Kalau di Wikidata namanya bukan 'kerajaanLabel', ganti dengan nama kolom pertama
    col_wiki = 'kerajaanLabel' if 'kerajaanLabel' in df_wiki.columns else df_wiki.columns[0]
    col_dbp = 'namaKerajaan' if 'namaKerajaan' in df_dbpedia.columns else df_dbpedia.columns[0]

    print(f"\n🔄 Menggabungkan {col_wiki} (Wiki) dengan {col_dbp} (DBpedia)...")

    df_wiki['kerajaan_key'] = df_wiki[col_wiki].astype(str).str.lower().str.strip()
    df_dbpedia['kerajaan_key'] = df_dbpedia[col_dbp].astype(str).str.lower().str.strip()

    df_final = pd.merge(df_wiki, df_dbpedia, on='kerajaan_key', how='left')
    df_final = df_final.drop(columns=['kerajaan_key'])

    print("✅ BERHASIL TOTAL! Data sudah dijahit.")
    display(df_final.head())

    # Simpan Hasil
    df_final.to_csv('data_gabungan_final.csv', index=False)
    print("💾 File 'data_gabungan_final.csv' siap didownload.")

except Exception as e:
    print(f"{e}")
    print("Cek manual kolom Wiki:", df_wiki.columns)
    print("Cek manual kolom DBpedia:", df_dbpedia.columns)

--- Memproses Wikidata ---
Kolom Wikidata yang ditemukan: ['kerajaanLabel', 'orangLabel', 'peranLabel', 'ayahLabel', 'ibuLabel', 'pasanganLabel', 'anakLabel']

--- Memproses DBpedia ---
Kolom DBpedia yang ditemukan: ['namaKerajaan', 'ibuKota', 'agama', 'pendahulu', 'penerus', 'tahunMulai', 'wikidataID']

🔄 Menggabungkan kerajaanLabel (Wiki) dengan namaKerajaan (DBpedia)...
✅ BERHASIL TOTAL! Data sudah dijahit.


,kerajaanLabel,orangLabel,peranLabel,ayahLabel,ibuLabel,pasanganLabel,anakLabel,namaKerajaan,ibuKota,agama,pendahulu,penerus,tahunMulai,wikidataID
0,Kesultanan Banjar,Soeria Kasoema,pejabat pemerintahan,,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kerajaan Klungkung,Dewa Agung Jambe I,,,,,Dewa Agung Gede,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kerajaan Klungkung,Dewa Agung Jambe II,,Dewa Agung Putra III Bhatara Dalem,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Kerajaan Klungkung,Dewa Agung Putra I Kusamba,,Dewa Agung Åšakti,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Kerajaan Klungkung,Dewa Agung Putra III Bhatara Dalem,,,,,Dewa Agung Jambe II,NaN,NaN,NaN,NaN,NaN,NaN,NaN


💾 File 'data_gabungan_final.csv' siap didownload.
